# Basis-Separating Counterexamples v0.12
Diagnostic study of the proposed systemhood coordinates B,C,A,P. No temporal residual is fitted here. Goal: determine whether the four coordinates can represent independent physical directions.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
names=['B','C','A','P']
# deliberately complementary toy regimes; values are normalized diagnostics, not measured constants
cases={
 'free-control':       [0.03,0.03,0.08,0.10],
 'entangled-unbound':  [0.03,0.95,0.18,0.48],
 'bound-weakcorr':     [0.92,0.28,0.82,0.90],
 'open-persistent':    [0.28,0.55,0.48,0.96],
 'autonomous-unbound': [0.08,0.18,0.92,0.78],
 'correlated-leaky':   [0.20,0.82,0.22,0.42],
 'fragile-bound':      [0.78,0.25,0.38,0.30],
 'persistent-lowbind': [0.12,0.32,0.62,0.94],
}
Z=np.array(list(cases.values()),float); labels=list(cases)
print('rank=',np.linalg.matrix_rank(Z)); s=np.linalg.svd(Z-Z.mean(0),compute_uv=False); print('centered singular values=',s); print('centered condition=',s[0]/s[-1])


## Correlation and geometry

In [ ]:
R=np.corrcoef(Z,rowvar=False); print(np.round(R,3))
plt.figure(figsize=(6,5)); plt.imshow(R,vmin=-1,vmax=1); plt.xticks(range(4),names); plt.yticks(range(4),names); plt.colorbar(label='correlation'); plt.tight_layout(); plt.show()


## Leave-one-feature-out reconstruction
If one coordinate is nearly reconstructible from the other three across diverse regimes, it may be redundant.

In [ ]:
for j,name in enumerate(names):
    keep=[k for k in range(4) if k!=j]; X=np.column_stack([np.ones(len(Z)),Z[:,keep]]); b=np.linalg.lstsq(X,Z[:,j],rcond=None)[0]; pred=X@b; ssr=np.sum((Z[:,j]-pred)**2); sst=np.sum((Z[:,j]-Z[:,j].mean())**2); r2=1-ssr/sst; print(name,'R2 from other three=',r2)


## Ablation of counterexample families
Check whether full rank depends on one hand-picked row.

In [ ]:
for i,label in enumerate(labels):
    Za=np.delete(Z,i,axis=0); sc=np.linalg.svd(Za-Za.mean(0),compute_uv=False); print('drop',label,' rank=',np.linalg.matrix_rank(Za),' centered smallest sv=',sc[-1])


## Random perturbation robustness
Toy scores are not exact. Perturb them and inspect how often the centered feature geometry becomes nearly singular.

In [ ]:
rng=np.random.default_rng(23); cond=[]; mins=[]
for _ in range(5000):
    Q=np.clip(Z+rng.normal(0,.04,Z.shape),0,1); sv=np.linalg.svd(Q-Q.mean(0),compute_uv=False); cond.append(sv[0]/sv[-1]); mins.append(sv[-1])
print('condition median/95%=',np.median(cond),np.quantile(cond,.95)); print('smallest sv 5%=',np.quantile(mins,.05))


## PCA-style explained variance
Full rank does not guarantee equally useful axes. Inspect how much centered variance is carried by each singular direction.

In [ ]:
sv=np.linalg.svd(Z-Z.mean(0),compute_uv=False); frac=sv**2/np.sum(sv**2); print('variance fractions=',frac,' cumulative=',np.cumsum(frac))


## Interpretation
This deliberately diversified toy suite can make B,C,A,P full-rank, showing that four independent directions are mathematically possible. That does not prove the coordinates are fundamental. The result depends on whether each diagnostic can be operationally computed from actual physical models rather than assigned by hand. The next step must replace these illustrative scores with model-derived quantities case by case.